**Predictor schema for the revised benchmark.** This notebook only builds the encoded predictor file used by `train/cohorts.py` as the reference list of the 94 model inputs. It does not split the data and does not remove participants with incomplete predictors. Splitting, training-only imputation, tuning and evaluation are performed by `train/` and `inference/` (see `README.md`). Run `all_y.ipynb` first.

In [ ]:
import warnings
from sklearn.exceptions import ConvergenceWarning
warnings.filterwarnings("ignore", category=ConvergenceWarning)
from config import *
from sklearn.utils import shuffle
import os
import re
import pandas as pd
import numpy as np
import pandas as pd
# Show all rows
pd.set_option('display.max_rows', None)
# Show all columns
pd.set_option('display.max_columns', None)
# Do not truncate column values
pd.set_option('display.max_colwidth', None)

In [ ]:
base_dir = 'data/preprocessed data'

In [ ]:
years = ['2000', '2002', '2004', '2006', '2008', '2010', '2012', '2014', '2016', '2018','2020']

In [ ]:
df = dict()
for year in years:
    df[year] = pd.read_csv(os.path.join(base_dir, f'{year}.csv'))

In [ ]:
for year in years:
    # Clean column names: remove '_' and replace '/' with ' or '
    cleaned_cols = [col.replace('_', '').replace('/', ' or ').replace('.', ' ').replace('-', 'to').replace('+', 'above').replace('0', '').replace(',', ' ')  for col in df[year].columns]
    df[year].columns = cleaned_cols

In [ ]:
df['2000'].describe()

In [ ]:
# Load the CSV into a DataFrame
all_y = pd.read_csv('data/preprocessed data/all_y.csv')
all_y.head(10)

In [ ]:
X_cols = [
'hhidpn', 'child', 'lbrf2 works pt', 'lbrf3 unemployed', 'lbrf4 partly retired', 'lbrf5 retired', 'lbrf6 disabled', 'lbrf7 not in lbrf', 'shlt', 'ageym', 'height', 'weight', 'smokev', 'proxy', 'effort', 'hibpe', 'diabe', 'vigact', 'slfmem', 'livpar', 'momage', 'dadage', 'livsib', 'hlthlm', 'hosp', 'nrshom', 'nrstim', 'nrsnit', 'doctim', 'depres', 'sleepr', 'whappy', 'flone', 'fsad', 'going', 'enlife', 'drink', 'smoken', 'cancre', 'lunge', 'hearte', 'stroke', 'arthre', 'toilta', 'adl5a', 'mapa', 'walksa', 'walk1a', 'sita', 'chaira', 'climsa', 'clim1a', 'stoopa', 'lifta', 'dimea', 'armsa', 'pusha', 'mobila', 'lgmusa', 'grossa', 'finea', 'walksadontdo', 'walk1adontdo', 'sitadontdo', 'chairadontdo', 'climsadontdo', 'clim1adontdo', 'stoopadontdo', 'armsadontdo', 'pushadontdo', 'liftadontdo', 'dimeadontdo', 'mapadontdo', 'gender', 'demcls', 'edstg2 8to11', 'edstg3 12', 'edstg4 13above', 'cendiv2 mid atlantic', 'cendiv3 en central', 'cendiv4 wn central', 'cendiv5 s atlantic', 'cendiv6 es central', 'cendiv7 ws central', 'cendiv8 mountain', 'cendiv9 pacific', 'cendiv11 not us or inc us terr', 'mstat2 married spouse absent', 'mstat3 partnered', 'mstat4 separated', 'mstat5 divorced', 'mstat7 widowed', 'mstat8 never married', 'raceeth1 ', 'raceeth2 ', 'raceeth3 '
]

# Complete-case status is defined on predictors only. hhidpn is an ID and demcls is the baseline
# dementia status used for eligibility; neither is a model input.
PREDICTORS = [c for c in X_cols if c not in ('hhidpn', 'demcls')]

***6 year interval***

In [ ]:
# Load feature columns for 2000 baseline train set
df['2000'] = df['2000'][X_cols] 
df['2000']=df['2000'].merge(all_y[['hhidpn','demcls00-06']].dropna(),on='hhidpn',how='inner')
df['2000'].rename(columns={'demcls00-06':'y'}, inplace=True)
df['2000'].to_csv('data/preprocessed data/2000(00-06y).csv',na_rep="NA" ,index=False)
df['2000'].head()

In [ ]:
# List of variables to check
var_list = [
    'child', 'lbrf2 works pt', 'lbrf3 unemployed', 'lbrf4 partly retired', 'lbrf5 retired', 'lbrf6 disabled', 'lbrf7 not in lbrf', 'shlt', 'ageym', 'height', 'weight', 'smokev', 'proxy', 'effort', 'hibpe', 'diabe', 'vigact', 'slfmem', 'livpar', 'momage', 'dadage', 'livsib', 'hlthlm', 'hosp', 'nrshom', 'nrstim', 'nrsnit', 'doctim', 'depres', 'sleepr', 'whappy', 'flone', 'fsad', 'going', 'enlife', 'drink', 'smoken', 'cancre', 'lunge', 'hearte', 'stroke', 'arthre', 'toilta', 'adl5a', 'mapa', 'walksa', 'walk1a', 'sita', 'chaira', 'climsa', 'clim1a', 'stoopa', 'lifta', 'dimea', 'armsa', 'pusha', 'mobila', 'lgmusa', 'grossa', 'finea', 'walksadontdo', 'walk1adontdo', 'sitadontdo', 'chairadontdo', 'climsadontdo', 'clim1adontdo', 'stoopadontdo', 'armsadontdo', 'pushadontdo', 'liftadontdo', 'dimeadontdo', 'mapadontdo', 'gender', 'edstg2 8to11', 'edstg3 12', 'edstg4 13above', 'cendiv2 mid atlantic', 'cendiv3 en central', 'cendiv4 wn central', 'cendiv5 s atlantic', 'cendiv6 es central', 'cendiv7 ws central', 'cendiv8 mountain', 'cendiv9 pacific', 'cendiv11 not us or inc us terr', 'mstat2 married spouse absent', 'mstat3 partnered', 'mstat4 separated', 'mstat5 divorced', 'mstat7 widowed', 'mstat8 never married', 'raceeth1 ', 'raceeth2 ', 'raceeth3 '
]

# Subset your dataframe
df_sub = df['2000'][var_list].copy()

# Define classification: treat as continuous if numeric & >10 unique values
continuous_vars = []
categorical_vars = []
for col in var_list:
    if col in df_sub.columns:
        s = df_sub[col]
        if pd.api.types.is_numeric_dtype(s) and s.nunique(dropna=True) > 10:
            continuous_vars.append(col)
        else:
            categorical_vars.append(col)

# Summarize continuous variables
cont_summary = []
for col in continuous_vars:
    if col in df_sub.columns:
        s = df_sub[col]
        cont_summary.append({
            "Variable": col,
            "Count": s.count(),
            "Mean": s.mean(),
            "Std": s.std(),
            "Min": s.min(),
            "Median": s.median(),
            "Max": s.max(),
            "Missing values": s.isnull().sum()
        })
cont_summary_df = pd.DataFrame(cont_summary)

# Summarize categorical variables
cat_summary_list = []
for col in categorical_vars:
    if col in df_sub.columns:
        vc = df_sub[col].value_counts(dropna=False)
        n = len(df_sub)
        for level, count in vc.items():
            pct = 100 * count / n
            cat_summary_list.append({
                "Variable": col,
                "Level": str(level),
                "Count (Percent)": f"{count} ({pct:.2f}%)"
            })
cat_summary_df = pd.DataFrame(cat_summary_list)

# Show output (if using Jupyter, these will display as nice tables)
print("Continuous variable summary:")
print(cont_summary_df)
print("\n Non-Continuous (Categorical & Boolean) variable summary:")
print(cat_summary_df)

In [ ]:
df['2000'].to_csv('data/preprocessed data/2000(00-06y)encode.csv', index=False)

In [ ]:
# Eligibility, applied identically to the 2000 development and 2010 external cohorts:
# aged > 50 at baseline and assessed dementia-free at baseline (demcls == 0; missing demcls is excluded).
def apply_eligibility(cohort, label):
    age_ok = cohort['ageym'] > 50
    dem_free = cohort['demcls'] == 0
    print(f"{label}: {len(cohort)} with an outcome | excluded age <= 50 or missing: {(~age_ok).sum()} | "
          f"excluded (age > 50) for baseline dementia: {(age_ok & (cohort['demcls'] == 1)).sum()}, "
          f"missing baseline dementia status: {(age_ok & cohort['demcls'].isna()).sum()} | "
          f"eligible: {(age_ok & dem_free).sum()}")
    return cohort[age_ok & dem_free].copy()

# preprocess_2000.ipynb already applies this rule to 2000, so this is a check that it holds.
df['2000'] = apply_eligibility(df['2000'], '2000 cohort')

# The cohort file used by the model notebooks is written in the previous cell, before this rule;
# re-save it with eligible participants only.
df['2000'].to_csv('data/preprocessed data/2000(00-06y)encode.csv', index=False)